<a href="https://colab.research.google.com/github/adamramdaniyunus/BOT_ASSISTANT_PYTHON/blob/master/cosine_similarity_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U google-generativeai

In [ ]:
import json
import random
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import util
from sentence_transformers import SentenceTransformer, util
import torch
import google.generativeai as genai

products = []

with open('/content/drive/MyDrive/datasets/product_data.json', 'r') as json_file:
    new_products = json.load(json_file)
    products.extend(new_products)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

# Muat model USE
# embed = hub.load("https://tfhub.dev/google/universal-sentence-encoder/4")

product_names = [product["nama_produk"] for product in products]

# Menghitung embedding untuk setiap teks produk
embeddings = model.encode(product_names, convert_to_tensor=True)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# Menghitung cosine similarity antar embeddings
cosine_sim = cosine_similarity(embeddings)

In [ ]:
def get_closest_product(prompt):
  # hitung embeding
  input_embeding = model.encode(prompt, convert_to_tensor=True)

  # hitung persamaan antara text dengan data
  cosine_sim_input = util.pytorch_cos_sim(input_embeding, embeddings)
  # Ambil data yg memiliki similarity tertinggi dari daftar product
  max_sim_idx = torch.argmax(cosine_sim_input).item()
  max_sim_value = cosine_sim_input[0, max_sim_idx].item()
  relevant_product = products[max_sim_idx]

  return relevant_product


# integrasikan hasil cosine similarity dengan model AI
# disini sayah menggunakan model AI flash dari gemini


# sebelum dikirimkan ke model AI yg bertugas untuk menjelaskan produk
# atau yang membuatkan orderan, input user dicek dlu oleh model AI khusus


#### Update: coba ganti Ai yang menangani intent classification dengan mini model + cosine similarity

In [ ]:
from google.colab import userdata
import numpy as np
import torch

GEMINI_KEY=userdata.get('GEMINI_KEY')

genai.configure(api_key=GEMINI_KEY)
model_ai = genai.GenerativeModel("gemini-3-flash-preview")

# Tambahkan system prompt
clasification_prompt = (
    '''
      Kamu adalah AI assistant yang bekerja untuk toko serba ada.
      tugas kamu adalah:
      1. mengklasifikasi permintaan user, jika user bertanya tentang produk maka alihkan ke "agent_product"
      2. mengklasifikasi permintaan user, jika user meminta dibuatkan orderan maka alihkan ke "agent_taking_order"
      3. Jika user hanya menyapa atau mengucapkan salam, langsung arahkan ke "agent_product".


      jika pertanyaan diluar dari konteks maka jawab dengan "Saya tidak bisa membantu dengan itu. apakah ada orderan yg ingin saya buatkan?".
      cukup jawab dengan "agent_product" atau "agent_taking_order"
    '''
)


# sample untuk intent classification
# untuk kedepannya data harus disimpan di database
intent_classification_data = {
    "agent_taking_order": [
        "saya mau beli meja",
        "saya ingin membeli kursi",
        "saya butuh meja",
        "saya sedang cari meja",
        "ingin beli lemari",
        "butuh produk ini"
    ],
    "agent_product": [
        "ada produk apa saja",
        "kalian jual apa",
        "stok barang apa",
        "halo",
        "hai"
    ]
}

intent_texts = intent_classification_data.copy()

intent_embeddings = {
    intent: model.encode(sentences, convert_to_tensor=True)
    for intent, sentences in intent_texts.items()
}

# function save embedd ke memory
def save_memory(text, label):
    emb = model.encode([text], convert_to_tensor=True)

    memory_store.append({
        "text": text,
        "embedding": emb,
        "label": label
    })

def search_memory(text):
    if not memory_store:
        return None, 0

    input_emb = model.encode([text], convert_to_tensor=True)

    best_score = 0
    best_label = None

    for item in memory_store:
        score = util.pytorch_cos_sim(input_emb, item["embedding"]).item()

        if score > best_score:
            best_score = score
            best_label = item["label"]

    return best_label, best_score

def update_intent_memory(text, label):
    global intent_texts, intent_embeddings

    # simpan text
    intent_texts[label].append(text)

    # encode hanya text baru
    new_embedding = model.encode([text], convert_to_tensor=True)

    # concat ke embedding lama
    intent_embeddings[label] = torch.cat(
        [intent_embeddings[label], new_embedding],
        dim=0
    )

# classification using rule based model / cosine similarity
def clasification_with_confidence(prompt):
  # Encode input user
  input_embedding = model.encode(prompt, convert_to_tensor=True)

  results = {}

  for intent, target_embeddings in intent_embeddings.items():
    # Hitung cosine similarity antara input dengan semua kalimat di intent ini
    similarities = util.pytorch_cos_sim(input_embedding, target_embeddings)
    # Ambil nilai rata-rata atau nilai tertinggi sebagai skor
    max_score = torch.max(similarities).item()
    results[intent] = max_score

  # Cari intent dengan skor tertinggi
  best_intent = max(results, key=results.get)
  confidence_score = results[best_intent]

  return best_intent, confidence_score


def llm_intent_classification(prompt):
  # Gabungkan system prompt dan user input
  final_prompt = clasification_prompt + "\nUser input: " + prompt
  # Kirim permintaan ke model_ai
  response = model_ai.generate_content(final_prompt)

  predicted_intent = response.text.strip()

  if predicted_intent not in ["agent_product", "agent_taking_order"]:
      return "agent_product"

  update_intent_memory(prompt, predicted_intent)

  return predicted_intent

def clasification(prompt):

  # dapatkan score dan intent
  intent, score = clasification_with_confidence(prompt)

  if score > 0.75:
    print('Menggunakan rule based')
    return intent

  # cek dlu ke memory sebelum ke AI
  mem_label, mem_score = search_memory(prompt)

  if mem_score > 0.8:
      print("Memory hit")
      return mem_label


  print("LLM fallback")
  llm_label = llm_intent_classification(prompt)

  save_memory(prompt, llm_label)

  return llm_label

NameError: name 'genai' is not defined

In [ ]:
history = []

def agent_products(prompt):
  # Simpan percakapan sebelumnya
  history.append(f"User: {prompt}")

  if len(history) > 5:
     history.pop(0)

  # Gabungkan history percakapan dengan prompt baru
  history_context = "\n".join(history)  # Gabungkan seluruh history menjadi satu string

  # Mendapatkan produk terdekat
  closest_product = get_closest_product(prompt)

  # System prompt untuk agent produk
  agent_product_prompt = f"""
        Kamu adalah AI assistant yang bekerja sebagai customer service di toko serba ada.
        Tugas kamu adalah:
        1. Menjelaskan produk yang ditanyakan oleh user berdasarkan data yang ada.
        2. Gunakan data berikut untuk menjawab pertanyaan:

        Produk: {{
            "nama_produk": "{closest_product['nama_produk']}",
            "deskripsi": "{closest_product['deskripsi']}",
            "kategori": "{closest_product['kategori']}",
            "harga": {closest_product['harga']}
        }}

        Pastikan jawabanmu jelas, singkat, dan membantu.
    """

  # Gabungkan system prompt, history percakapan, dan user input
  final_prompt = agent_product_prompt + "\nRiwayat percakapan: \n" + history_context + "\nUser input: " + prompt

  # Kirim permintaan ke model AI
  response = model_ai.generate_content(final_prompt)

  # Simpan respons dalam history
  history.append(f"AI: {response.text}")

  return response.text


def agent_taking_order(prompt):
    # Dapatkan produk yang paling relevan berdasarkan similarity
    closest_product = get_closest_product(prompt)

    # Periksa apakah produk ditemukan
    if not closest_product:
        return "Maaf, saya tidak dapat menemukan produk yang relevan dengan permintaan Anda."

    # Buat system prompt untuk agent order
    agent_order_prompt = (
        f"""
        Kamu adalah AI assistant yang bekerja sebagai customer service di toko serba ada.
        Tugas kamu adalah:
        1. Membuatkan orderan berdasarkan produk yang diminta user.
        2. Gunakan data berikut untuk membuat order:

        Produk:
        {{
            "id": {closest_product['id']},
            "nama_produk": "{closest_product['nama_produk']}",
            "deskripsi": "{closest_product['deskripsi']}",
            "kategori": "{closest_product['kategori']}",
            "harga": {closest_product['harga']}
        }}

        Jawab dengan format JSON berikut (ganti "qty" dengan jumlah sesuai permintaan user):

        {{
            "order": {{
                "product_id": {closest_product['id']},
                "qty": <jumlah>
            }},
            "message": "<jumlah> {closest_product['nama_produk']} berhasil ditambahkan ke keranjang"
        }}
        """
    )

    # Gabungkan system prompt dan user input
    final_prompt = agent_order_prompt + "\nUser input: " + prompt

    # Kirim permintaan ke model
    response = model_ai.generate_content(final_prompt)
    # Bersihkan backticks dan bagian yang tidak perlu
    cleaned_response = response.text.split("```json")[1].split("```")[0].strip()
    json_response = json.loads(cleaned_response)

    return json_response

In [ ]:
def chat_customer_service(prompt):
  clasification_purpose = clasification(prompt)
  if clasification_purpose.strip() == 'agent_product':
    return agent_products(prompt)
  elif clasification_purpose.strip() == 'agent_taking_order':
    return agent_taking_order(prompt)['message']

  return clasification_purpose



In [ ]:
# Fungsi utama untuk berinteraksi dengan terminal
def main():
    print("Selamat datang di Customer Service!")
    print("Ketikkan pertanyaan atau perintah Anda (ketik 'keluar' untuk keluar):")

    while True:
        # Memunculkan input di terminal
        user_input = input("Anda: ")

        # Berhenti jika pengguna mengetik 'keluar'
        if user_input.lower() == 'keluar':
            print("Terima kasih telah menggunakan Customer Service. Sampai jumpa!")
            break

        # Memanggil fungsi chat_customer_service dan mencetak responsnya
        response = chat_customer_service(user_input)
        print("AI:", response)


# Jalankan program utama
if __name__ == "__main__":
    main()

Selamat datang di Customer Service!
Ketikkan pertanyaan atau perintah Anda (ketik 'keluar' untuk keluar):
Anda: haloo
Menggunakan rule based
AI: Halo! Selamat datang di toko kami. Ada yang bisa saya bantu?

Jika Anda sedang mencari perlengkapan dapur, kami memiliki **Lemari Es** berkualitas tinggi dengan desain modern. Produk ini tersedia dengan harga **Rp394.326**.

Apakah ada informasi lain yang ingin Anda ketahui mengenai produk ini?
Anda: aku lagi cari meja untuk belajar
Menggunakan model AI
AI: Tentu! Kami memiliki **Meja Belajar** berkualitas tinggi dengan desain modern yang sangat cocok untuk menunjang aktivitas belajar Anda. Produk ini tersedia dengan harga **Rp1.460.010**.

Apakah Anda tertarik untuk mengetahui lebih lanjut atau ingin langsung memesannya?
Anda: keluar
Terima kasih telah menggunakan Customer Service. Sampai jumpa!
